In [509]:
# https://deeplearningcourses.com/c/artificial-intelligence-reinforcement-learning-in-python
# https://www.udemy.com/artificial-intelligence-reinforcement-learning-in-python
from __future__ import print_function, division
from builtins import range
# Note: you may need to update your version of future
# sudo pip install -U future

In [510]:
import numpy as np
import matplotlib.pyplot as plt

ACTIONS = ['HIT', 'STICK']
STATES = list(range(4, 21))  # player sum from 4 to 20
GAMMA = 1.0  # no discounting in episodic MC

class BlackjackEnv:
    def __init__(self):
        self.record = []
        self.sum = 0
        self.stick = False

    def draw_card(self):
        # Draw a card, 1-10, where 1 is Ace (11), 2-10 are face values
        card = min(10, np.random.randint(1, 15))
        if card == 1:
            card = 11
        self.record.append(card)
        self.sum += card
        while self.sum > 21 and 11 in self.record:
            self.record[self.record.index(11)] = 1
            self.sum -= 10
        
    def is_terminal(self):
        return self.sum >= 21 or self.stick

    def reset(self):
        # Deal two cards initially
        self.sum = 0
        self.stick = False
        self.draw_card()
        self.draw_card()
        return self.sum

    def step(self, action): # Returns state, reward, done
        if action == 'HIT':
            self.draw_card()
            if self.sum > 21:
                return self.sum, -1, True  # sum, reward, whether stop (bust)
            elif self.sum == 21:
                return self.sum, 1, True
            else:
                return self.sum, 0, False
        elif action == 'STICK':
            self.stick = True
            return self.sum, self.sum / 21.0, True  # sum, reward based on final sum, stop
        else:
            raise ValueError("Unknown action")

def max_dict(d):
    if not d:
        return None
    max_key = max(d, key=d.get)
    return max_key, d[max_key]

def epsilon_greedy(s, Q, epsilon=0.1):
    if np.random.rand() < epsilon:
        return np.random.choice(ACTIONS)
    else:
        return max_dict(Q[s])[0]

def play_game(policy):
    env = BlackjackEnv()
    state = env.reset()
    done = env.is_terminal()
    states, rewards, actions = [state], [0], []
    while not done:
        action = policy(state)
        state, reward, done = env.step(action)
        states.append(state)
        rewards.append(reward)
        actions.append(action)
    actions.append(None)
    return states, rewards, actions

def MC_control(Q, counts, runs=100, gamma=0.9, epsilon=0.1):
    for _ in range(runs):
        states, rewards, actions = play_game(lambda s: epsilon_greedy(s, Q, epsilon))
        G = 0
        for t in range(len(states) - 2, -1, -1):
            G = gamma * G + rewards[t + 1]
            counts[states[t]][actions[t]] += 1
            Q[states[t]][actions[t]] += 1 / counts[states[t]][actions[t]] * (G - Q[states[t]][actions[t]])

# It doesn't work because it doesn't distinguish a state where you already sticked from a state where you can hit with equal sum (19 but already sticked so terminal state, or 19 and you can stick or hit so not terminal, and you can get stick when you pass from one to another)
def Q_learning_count(Q, counts, runs=100, gamma=0.9, epsilon=0.1):
    env = BlackjackEnv()
    for _ in range(runs):
        state = env.reset()
        done = env.is_terminal()
        while not done:
            action = epsilon_greedy(state, Q, epsilon)
            next_state, reward, done = env.step(action)
            counts[state][action] += 1
            alpha = 1.0 / counts[state][action]
            Q[state][action] += alpha * (reward + gamma * (max_dict(Q[next_state])[1] if not done else 0) - Q[state][action])
            state = next_state

def Q_learning(Q, runs=100, alpha=0.1, gamma=0.9, epsilon=0.1):
    env = BlackjackEnv()
    for _ in range(runs):
        state = env.reset()
        done = env.is_terminal()
        while not done:
            action = epsilon_greedy(state, Q, epsilon)
            next_state, reward, done = env.step(action)
            Q[state][action] += alpha * (reward + gamma * (max_dict(Q[next_state])[1] if not done else 0) - Q[state][action])
            state = next_state

In [511]:
# Initialize randomly
# Q = {s: {a: np.random.rand() for a in ACTIONS} for s in STATES}
Q = {s: {a: 0 for a in ACTIONS} for s in STATES}
counts = {s: {a: 0 for a in ACTIONS} for s in STATES}
policy = {s: max_dict(Q[s])[0] for s in STATES}

print("Starting policy")
for s in range(4, 21):
    print(f"State {s}: {policy[s]}")

# Print Q as a table
for s in range(4, 21):
    print(f"State {s}: HIT={Q[s]['HIT']:.2f}, STICK={Q[s]['STICK']:.2f}")

# Print counts as a table
for s in range(4, 21):
    print(f"State {s}: HIT={counts[s]['HIT']}, STICK={counts[s]['STICK']}")

Starting policy
State 4: HIT
State 5: HIT
State 6: HIT
State 7: HIT
State 8: HIT
State 9: HIT
State 10: HIT
State 11: HIT
State 12: HIT
State 13: HIT
State 14: HIT
State 15: HIT
State 16: HIT
State 17: HIT
State 18: HIT
State 19: HIT
State 20: HIT
State 4: HIT=0.00, STICK=0.00
State 5: HIT=0.00, STICK=0.00
State 6: HIT=0.00, STICK=0.00
State 7: HIT=0.00, STICK=0.00
State 8: HIT=0.00, STICK=0.00
State 9: HIT=0.00, STICK=0.00
State 10: HIT=0.00, STICK=0.00
State 11: HIT=0.00, STICK=0.00
State 12: HIT=0.00, STICK=0.00
State 13: HIT=0.00, STICK=0.00
State 14: HIT=0.00, STICK=0.00
State 15: HIT=0.00, STICK=0.00
State 16: HIT=0.00, STICK=0.00
State 17: HIT=0.00, STICK=0.00
State 18: HIT=0.00, STICK=0.00
State 19: HIT=0.00, STICK=0.00
State 20: HIT=0.00, STICK=0.00
State 4: HIT=0, STICK=0
State 5: HIT=0, STICK=0
State 6: HIT=0, STICK=0
State 7: HIT=0, STICK=0
State 8: HIT=0, STICK=0
State 9: HIT=0, STICK=0
State 10: HIT=0, STICK=0
State 11: HIT=0, STICK=0
State 12: HIT=0, STICK=0
State 13: HI

In [512]:
# Learning
# Q_learning(Q, runs=10000, alpha=0.01, gamma=0.8, epsilon=0.5)
Q_learning_count(Q, counts, runs=10000, gamma=0.9, epsilon=0.2)
# MC_control(Q, counts, runs=10000, gamma=1, epsilon=0.2)

policy = {s: max_dict(Q[s])[0] for s in STATES}

print("Final policy")
for s in range(4, 21):
    print(f"State {s}: {policy[s]}")

# Print Q as a table
for s in range(4, 21):
    print(f"State {s}: HIT={Q[s]['HIT']:.2f}, STICK={Q[s]['STICK']:.2f}")

# Print counts as a table
for s in range(4, 21):
    print(f"State {s}: HIT={counts[s]['HIT']}, STICK={counts[s]['STICK']}")

Final policy
State 4: HIT
State 5: HIT
State 6: HIT
State 7: HIT
State 8: HIT
State 9: HIT
State 10: HIT
State 11: HIT
State 12: HIT
State 13: HIT
State 14: STICK
State 15: STICK
State 16: STICK
State 17: STICK
State 18: STICK
State 19: STICK
State 20: STICK
State 4: HIT=0.62, STICK=0.19
State 5: HIT=0.63, STICK=0.24
State 6: HIT=0.67, STICK=0.29
State 7: HIT=0.67, STICK=0.33
State 8: HIT=0.71, STICK=0.38
State 9: HIT=0.73, STICK=0.43
State 10: HIT=0.77, STICK=0.48
State 11: HIT=0.80, STICK=0.52
State 12: HIT=0.65, STICK=0.57
State 13: HIT=0.67, STICK=0.62
State 14: HIT=0.66, STICK=0.67
State 15: HIT=0.68, STICK=0.71
State 16: HIT=0.66, STICK=0.76
State 17: HIT=0.70, STICK=0.81
State 18: HIT=0.73, STICK=0.86
State 19: HIT=0.71, STICK=0.90
State 20: HIT=0.72, STICK=0.95
State 4: HIT=64, STICK=10
State 5: HIT=85, STICK=20
State 6: HIT=154, STICK=19
State 7: HIT=209, STICK=17
State 8: HIT=232, STICK=33
State 9: HIT=302, STICK=31
State 10: HIT=384, STICK=38
State 11: HIT=446, STICK=42
Stat